# TEKNOFEST 2026 — Variant Pathogenicity Classifier
## Phases 7 → 10  (Threshold Optimisation · SHAP · Ablation · Tracking & Packaging)

Colab **driver** for the final stage of the pipeline. Phases 0–6 already produced the
materialised feature matrices, the persisted CV folds, the base-model **OOF** probabilities,
the calibrated **stacked** meta-learner, and the **canary** predictions. Every heavy routine
here lives in the importable `src/` package, so this notebook only orchestrates.

**Inputs (from Phases 0–6):**
- `data/processed/development_features.parquet` / `canary_features.parquet` — features + `Label` + `source`, **`NaN` preserved**.
- `outputs/folds/fold_indices.json` — the 5-fold `StratifiedGroupKFold` indices.
- `outputs/feature_selection/selected_features.json` — the Phase-5 selected subset.
- `outputs/oof_predictions/*_oof.npy` + `*_extra.npy` and `outputs/models/stacking/stacked_oof.npy` — the Level-0/Level-1 predictions (loaded if present; otherwise transparently rebuilt with default params).

**Scope (Phases 7 → 10 only):**
- **Phase 7 — Threshold Optimisation** (`src/threshold_optimizer.py`): multi-metric search (**F1, MCC, Youden's J** simultaneously) on the stacked OOF, plus **per-dataset** optimal thresholds for `master/kanser/pah/cftr`. Saves `outputs/models/thresholds.json`.
- **Phase 8 — SHAP & Interpretability** (`src/shap_analysis.py`): global beeswarm, **per-dataset** panels (PAH vs CFTR), top-5 **dependence** plots, and **calibration curves** (before/after Isotonic). All saved to `outputs/plots/`.
- **Phase 9 — Ablation** (`src/experiment_tracking.py`): cumulative feature-group ablation through the shared CV loop, tracking **per-source** AUC and logging every run.
- **Phase 10 — Tracking & Packaging** (`src/experiment_tracking.py`): append a structured row to `outputs/experiment_log.csv` (per-source CV AUC, chosen thresholds, canary AUC), then package the inference-ready pipeline manifest.

> **Constraints honoured:** no `!pip install` (Colab ships everything); GPU params come **only** from `config.model_gpu_params()`; tree models receive `NaN` **natively**; folds and thresholds are **loaded**, never silently regenerated.

In [ ]:
# --- Environment setup -------------------------------------------------------
# Resolve the repo root (so ./data and ./outputs paths work) and put it on
# sys.path so `from src import ...` works whether launched from ./ or ./notebooks.
import os
import sys
import json


def find_repo_root(start: str = None) -> str:
    p = start or os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "data")) and os.path.isdir(os.path.join(p, "src")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return start or os.getcwd()


ROOT = find_repo_root()
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score

from src import config as C
from src import cv_runner as cvr
from src import stacking as stk
from src import threshold_optimizer as topt
from src import shap_analysis as sa
from src import experiment_tracking as et

C.set_global_seed(C.SEED)
C.ensure_dirs()

# GPU parameters come EXCLUSIVELY from the centralized config (project constraint).
USE_GPU = C.detect_gpu()

print("Repo root     :", os.getcwd())
print("Python        :", sys.version.split()[0])
print("GPU available :", USE_GPU)
print("Plots dir     :", C.PLOTS_DIR)
print("Thresholds    :", C.THRESHOLDS_PATH)
print("Experiment log:", C.EXPERIMENT_LOG_PATH)

## Load Matrices, Folds, Selected Features

`cv_runner.load_development_matrix()` splits `Label`/`source` off the materialised parquet and
returns `X` with **`NaN` preserved** (`float32`, no imputation). `load_folds()` reads the **exact**
`fold_indices.json` from Phase 0.4. The Phase-5 `selected_features.json` is loaded if present
(falling back to all features), so every downstream step operates on the same feature space the
stacked model was built on.

In [ ]:
# --- Load development / canary matrices, folds, and the selected feature subset
X, y, source = cvr.load_development_matrix()       # NaN preserved; Label/source split off
X_canary, y_canary, source_canary = cvr.load_canary_matrix()
folds = cvr.load_folds()
ALL_FEATURES = cvr.feature_columns(X)

if os.path.exists(C.SELECTED_FEATURE_LIST_PATH):
    SELECTED = json.load(open(C.SELECTED_FEATURE_LIST_PATH))["selected_features"]
else:
    SELECTED = ALL_FEATURES
    print("WARNING: selected_features.json not found -> falling back to ALL features.")
SELECTED = [c for c in SELECTED if c in X.columns]   # guard against drift

y_np = np.asarray(y).astype(int)
print(f"development X = {X.shape} | canary X = {X_canary.shape} | folds = {len(folds)}")
print(f"selected features = {len(SELECTED)} / {len(ALL_FEATURES)}")
print("per-source rows (development):", pd.Series(source).value_counts().to_dict())

## Acquire Base OOF, Stacked OOF & Fold Models

Phase 7 needs the **stacked OOF probabilities**; Phase 8 (SHAP) additionally needs **fitted fold
models** of the best base learner. The cell below is *load-or-rebuild*:

1. **Load** the persisted Phase-4/6 artefacts (`*_oof.npy`, `*_extra.npy`, `stacked_oof.npy`) — the
   exact, Optuna-tuned predictions. Then refit **only** the single best base model (cheap, 5 folds)
   to recover the fold models SHAP requires (Phase 6 saved OOF but not the per-fold models).
2. **Rebuild** everything (base CV + stacking) if those artefacts are incomplete — using `best_params.json`
   when present, otherwise the plan's strong defaults. This keeps the notebook runnable end-to-end while
   reusing the identical, leakage-free CV/stacking code from Phases 4–6.

Either way the result is a consistent `(oof_dict, canary_dict, stacked_oof, shap_cv)` quadruple.

In [ ]:
# --- Load-or-rebuild the Level-0 OOF + Level-1 stacked OOF + SHAP fold models ----
BASE_NAMES = ["XGB_tuned", "LGBM_tuned", "CatBoost_tuned", "Skeleton_head", "Dense_head"]
MAIN_MODELS = ["XGB_tuned", "LGBM_tuned", "CatBoost_tuned"]
_PKEY = {"XGB_tuned": "xgb", "LGBM_tuned": "lgbm", "CatBoost_tuned": "cat"}
_MAKE = {"XGB_tuned": cvr.make_xgb_learner, "LGBM_tuned": cvr.make_lgbm_learner,
         "CatBoost_tuned": cvr.make_catboost_learner}


def _load_npy(p):
    return np.load(p) if os.path.exists(p) else None


def _load_json(p):
    return json.load(open(p)) if os.path.exists(p) else None


best_params = _load_json(os.path.join(C.MODELS_DIR, "best_params.json")) or {}

# (a) try persisted base OOF + canary preds + stacked OOF
oof_dict, canary_dict = {}, {}
for nm in BASE_NAMES:
    o = _load_npy(os.path.join(C.OOF_DIR, f"{nm}_oof.npy"))
    e = _load_npy(os.path.join(C.OOF_DIR, f"{nm}_extra.npy"))
    if o is not None:
        oof_dict[nm] = o
    if e is not None:
        canary_dict[nm] = e
stacked_oof = _load_npy(os.path.join(C.STACKING_DIR, "stacked_oof.npy"))
canary_stacked = _load_npy(os.path.join(C.STACKING_DIR, "stacked_extra.npy"))
have_base = all(m in oof_dict for m in MAIN_MODELS)

if have_base and stacked_oof is not None:
    print("Loaded persisted base OOF + stacked OOF from Phases 4-6.")
    # refit ONLY the best main model to recover fold models for SHAP (cheap)
    best_main = max(MAIN_MODELS, key=lambda m: roc_auc_score(y_np, oof_dict[m]))
    shap_cv = cvr.run_cv(_MAKE[best_main](best_params.get(_PKEY[best_main]), y=y, use_gpu=USE_GPU),
                         X, y, source, folds, name=best_main, feature_cols=SELECTED,
                         keep_models=True, verbose=False)
else:
    print("Persisted OOF/stack incomplete -> rebuilding base models + stack "
          "(uses best_params.json if present, else plan defaults).")
    base_results = {}
    for nm in MAIN_MODELS:
        base_results[nm] = cvr.run_cv(
            _MAKE[nm](best_params.get(_PKEY[nm]), y=y, use_gpu=USE_GPU),
            X, y, source, folds, name=nm, feature_cols=SELECTED,
            X_extra=X_canary, keep_models=True, verbose=False)
    # Phase 4.6 missingness heads (XGB on tier-specific feature subsets)
    if "is_skeleton_row" in X.columns and "is_dense_row" in X.columns:
        for nm, tier, mask_col in [("Skeleton_head", "skeleton", "is_skeleton_row"),
                                   ("Dense_head", "dense", "is_dense_row")]:
            base_results[nm] = cvr.run_cv(
                cvr.make_xgb_learner(best_params.get("xgb"), y=y, use_gpu=USE_GPU),
                X, y, source, folds, name=nm,
                feature_cols=cvr.tier_feature_cols(SELECTED, tier),
                train_row_mask=(X[mask_col] == 1).to_numpy(),
                X_extra=X_canary, keep_models=False, verbose=False)
    oof_dict = {nm: r.oof for nm, r in base_results.items()}
    canary_dict = {nm: r.extra_pred for nm, r in base_results.items()}
    stack = stk.run_stacking(oof_dict, y, X, source=source, folds=folds,
                             extra_pred_dict=canary_dict, X_extra=X_canary, save=True)
    stacked_oof = stack.stacked_oof
    canary_stacked = stack.extra_pred
    shap_cv = base_results[max(MAIN_MODELS, key=lambda m: base_results[m].oof_metrics["auc"])]

print("\nbase models      :", list(oof_dict))
print("stacked OOF AUC  :", round(float(roc_auc_score(y_np, stacked_oof)), 4))
print("SHAP base model  :", shap_cv.name, f"({len(shap_cv.models)} fold models, "
      f"{len(shap_cv.feature_cols)} features)")

## Phase 7 — Threshold Optimisation

With ~2.7:1 imbalance (PAH ~5:1) the optimal decision boundary is **not** 0.5. `optimize_thresholds`:

1. **Multi-metric search (7.2)** — sweeps `np.arange(0.05, 0.95, 0.005)` and finds the optimum for
   **F1, MCC and Youden's J** *separately*, plus their mean as the ensemble candidate.
2. **Per-dataset thresholds (7.3)** — a separate optimum for `master / kanser / pah / cftr`, with the
   global threshold as the fallback. A single global cut systematically underperforms on PAH.

The result is saved to **`outputs/models/thresholds.json`** (the exact lookup test-time inference uses),
and the evaluation table quantifies the per-source F1/MCC gain over a single global threshold.

In [ ]:
# --- Phase 7: multi-metric + per-dataset thresholds on the stacked OOF -------
threshold_cfg, threshold_eval = topt.optimize_thresholds(
    y_np, stacked_oof, source, primary_metric="f1", save=True, path=C.THRESHOLDS_PATH)

g = threshold_cfg.global_
print("\nGlobal multi-metric optima:")
print(f"  F1       -> t={g['f1']['threshold']:<6} score={g['f1']['score']}")
print(f"  MCC      -> t={g['mcc']['threshold']:<6} score={g['mcc']['score']}")
print(f"  Youden J -> t={g['youden_j']['threshold']:<6} score={g['youden_j']['score']}")
print(f"  ensemble-mean t = {g['ensemble_mean']['threshold']}  | chosen (F1) = {g['chosen']}")

print("\nPer-source applied thresholds:", threshold_cfg.apply)
print("\nGlobal-threshold vs per-source-threshold (F1 / MCC):")
print(threshold_eval.to_string(index=False))

## Phase 8 — SHAP Analysis & Interpretability

`collect_oof_shap` explains the best base model's **fold models on their own validation rows** and
concatenates them — an out-of-fold SHAP matrix that mirrors the OOF predictions (leakage-free).
`run_shap_report` then writes the four report visuals to **`outputs/plots/`**:

- `shap_summary_global.png` — beeswarm of the top-30 features (Phase 8.1).
- `shap_summary_per_dataset.png` — per-source mean|SHAP| panels with **PAH** and **CFTR** highlighted (Phase 8.4) — PAH is expected to lean on conservation/EK scores, CFTR on AA physicochemical disruption.
- `shap_dependence_top5.png` — dependence plots for the 5 strongest features, interaction-coloured (Phase 8.2).
- `calibration_curves.png` — reliability diagram per base model, **before vs after Isotonic Regression** (Phase 8.5); calibrated OOF reuses the exact Phase-6 isotonic step.

In [ ]:
# --- Phase 8: compute OOF SHAP + write the report visuals -------------------
# Out-of-fold SHAP from the best base model's per-fold models (val rows only).
shap_bundle = sa.collect_oof_shap(
    shap_cv.models, X, folds, shap_cv.feature_cols, source=source, max_samples=4000)

# Calibrated OOF for the before/after-Isotonic calibration panel (exact Phase-6 step).
calibrated_oof, _ = stk.calibrate_oof(oof_dict, y_np, save=False)

shap_report = sa.run_shap_report(
    shap_bundle,
    oof_uncalibrated=oof_dict, oof_calibrated=calibrated_oof, y=y_np,
    out_dir=C.PLOTS_DIR, max_display=30, top_n_per_source=15, dependence_n=5)

In [ ]:
# --- Display the saved report visuals inline --------------------------------
from IPython.display import Image, display

for key in ("global", "per_dataset", "dependence", "calibration"):
    if key in shap_report.paths:
        print(f"\n=== {os.path.basename(shap_report.paths[key])} ===")
        display(Image(filename=shap_report.paths[key]))

## Phase 9 — Feature-Group Ablation

`feature_group_ablation` partitions the engineered features into the plan's groups (EK / AA /
missingness-meta / population-frequency / consensus / interactions) and runs the shared CV loop on
**cumulative** sets (`EK_only` → `EK+AA` → `+missing_meta` → `+pop_freq` → `+consensus` →
`all_engineered+interactions` → `all_features`). It tracks **per-source** AUC for each set and logs
every run to `experiment_log.csv`, giving the Phase-9 ablation table. A lighter `n_estimators=800`
keeps the sweep fast — this is for *relative* feature-group comparison, not final scoring.

In [ ]:
# --- Phase 9: cumulative feature-group ablation (logged to experiment_log.csv)
ablation_factory = cvr.make_xgb_learner({"n_estimators": 800}, y=y, use_gpu=USE_GPU)
ablation = et.feature_group_ablation(
    ablation_factory, X, y, source, folds,
    all_cols=ALL_FEATURES, model_type="xgb_ablation", log=True)

print("\nFeature-group ablation (per-source OOF AUC):")
print(ablation.to_string(index=False))

## Phase 10 — Experiment Tracking & Pipeline Packaging

Two final steps:

1. **Log the final stacked model** to `outputs/experiment_log.csv` — timestamp, git commit, **per-source
   CV AUC**, F1/MCC at the chosen threshold, the **global + per-source thresholds**, and the one-time
   **canary AUC** (`build_experiment_record` → `log_experiment`).
2. **Package the pipeline** (`package_pipeline`) — a manifest pointing at every artifact test-time
   inference needs (feature schema, EK normaliser, target-encoding maps, calibrators, meta-learner,
   thresholds), with existence checks, the ordered inference recipe, the RNG seed and library versions,
   so the model is **ready for test-set inference**.

In [ ]:
# --- Phase 10a: log the final stacked model to experiment_log.csv -----------
g_thr = threshold_cfg.apply["global"]
y_hat_global = (stacked_oof >= g_thr).astype(int)
stacked_per_source = cvr.per_source_auc(y_np, stacked_oof, source)
canary_auc = float(roc_auc_score(y_canary, canary_stacked)) if canary_stacked is not None else None

final_summary = {
    "cv_auc_mean": float(roc_auc_score(y_np, stacked_oof)),     # stacked OOF AUC
    "cv_f1_mean": float(f1_score(y_np, y_hat_global, zero_division=0)),
    "cv_mcc_mean": float(matthews_corrcoef(y_np, y_hat_global)),
}

record = et.build_experiment_record(
    model_type="ensemble_v2_stack",
    feature_set="selected_features_v1",
    n_features=len(SELECTED),
    encoding="target_enc(CAT_1/2,aa_pair,source) + onehot(geno/region) + native_nan",
    imbalance_method="scale_pos_weight + is_unbalance + Balanced",
    cv_summary=final_summary,
    per_source_auc=stacked_per_source,
    thresholds=threshold_cfg,
    canary_auc=canary_auc,
    cat_leakage_risk="low",
    pseudo_labels_used=False,
    notes="Phase 6 LR-stacked meta-learner; Phase 7 per-source thresholds.")

log_df = et.log_experiment(record)
print("\nfinal stacked model record:")
print(log_df.tail(1).T.to_string())
print(f"\ncanary AUC = {canary_auc} | stacked OOF AUC = {final_summary['cv_auc_mean']:.4f} | "
      f"gap = {abs((canary_auc or final_summary['cv_auc_mean']) - final_summary['cv_auc_mean']):.4f}")

In [ ]:
# --- Phase 10b: package the inference-ready pipeline manifest ---------------
stack_cfg = _load_json(os.path.join(C.STACKING_DIR, "stacking_config.json")) or {}
base_order = stack_cfg.get("base_order", list(oof_dict))
meta_cols = stack_cfg.get("meta_cols")

pkg = et.package_pipeline(
    selected_features=SELECTED,
    base_order=base_order,
    meta_cols=meta_cols,
    thresholds=threshold_cfg,
    metrics={"stacked_oof_auc": final_summary["cv_auc_mean"],
             "stacked_oof_auc_per_source": stacked_per_source,
             "canary_auc": canary_auc},
)

missing = et.verify_inference_ready(pkg.manifest)
print("\ninference_ready:", pkg.manifest["inference_ready"], "| missing artifacts:", missing)
print("\nordered inference recipe:")
for i, step in enumerate(pkg.manifest["inference_recipe"], 1):
    print(f"  {i:>2}. {step}")

In [ ]:
# --- Artifacts summary (end of Phase 10) ------------------------------------
print("Phases 7-10 complete. Persisted artifacts:\n")
expected = [
    C.THRESHOLDS_PATH,                                            # Phase 7
    os.path.join(C.PLOTS_DIR, "shap_summary_global.png"),        # Phase 8
    os.path.join(C.PLOTS_DIR, "shap_summary_per_dataset.png"),
    os.path.join(C.PLOTS_DIR, "shap_dependence_top5.png"),
    os.path.join(C.PLOTS_DIR, "calibration_curves.png"),
    os.path.join(C.PLOTS_DIR, "shap_importance_ranking.csv"),
    C.EXPERIMENT_LOG_PATH,                                        # Phase 9 + 10
    C.PIPELINE_MANIFEST_PATH,                                     # Phase 10
    os.path.join(C.FINAL_PIPELINE_DIR, "pipeline_state.joblib"),
]
for p in expected:
    print(f"  [{'OK ' if os.path.exists(p) else 'MISS'}] {p}")

print("\nFinal-stage outputs ready: per-source decision thresholds, the SHAP/calibration "
      "report visuals, the experiment log, and the inference-ready pipeline manifest.")